<a href="https://colab.research.google.com/github/Marjan-Akhtar2/Medical-Insurance-Premium-Prediction/blob/main/insurance_premium_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install -U scikit-learn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

In [27]:
df = pd.read_csv('/content/insurance_data.csv')
df

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
0,56,97.2,1.56,14.81,True,Mumbai,unemployed,High
1,69,109.4,1.51,9.66,True,Bangalore,retired,High
2,46,69.4,1.65,4.22,True,Mumbai,business_owner,Medium
3,32,61.5,1.64,6.53,False,Bangalore,private_job,Low
4,60,98.3,1.60,3.55,True,Bangalore,unemployed,High
...,...,...,...,...,...,...,...,...
95,42,64.1,1.64,20.01,False,Delhi,unemployed,Medium
96,62,67.1,1.79,20.62,True,Lucknow,retired,High
97,58,69.2,1.78,11.80,False,Mumbai,government_job,Low
98,46,108.7,1.55,20.61,False,Bangalore,government_job,Medium


In [28]:
dfFeat = df.copy()
dfFeat['bmi'] = dfFeat["weight"]/(dfFeat["height"]**2)


In [29]:
def age_group(age):
  if age < 20:
    return "young"
  elif age < 45:
    return "audult"
  elif age < 60:
    return "MiddleAge"
  return "old"



In [30]:
dfFeat["age_group"] = dfFeat["age"].apply(age_group)
def lifestyle_risk(row):
  if row["smoker"] and row["bmi"] > 30:
    return "high"
  elif row["smoker"] or row["bmi"] > 27:
    return "medium"

  else:
    return "low"


In [31]:
dfFeat["lifestyle_risk"]= dfFeat.apply(lifestyle_risk, axis=1)
tier_1_cities = ["Mumbai", "Delhi", "Bangalore"]
def city_tier(city):
    if city in tier_1_cities:
        return 1
    else:
        return 2

In [32]:
dfFeat["city_tier"] = dfFeat["city"].apply(city_tier)

In [33]:
dfFeat.drop(columns=["age","weight","height","smoker","city"])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']]

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
0,14.81,unemployed,39.940828,MiddleAge,high,1,High
1,9.66,retired,47.980352,old,high,1,High
2,4.22,business_owner,25.491276,MiddleAge,medium,1,Medium
3,6.53,private_job,22.865854,audult,low,1,Low
4,3.55,unemployed,38.398437,old,high,1,High
...,...,...,...,...,...,...,...
95,20.01,unemployed,23.832540,audult,low,1,Medium
96,20.62,retired,20.941918,old,medium,2,High
97,11.80,government_job,21.840677,MiddleAge,low,1,Low
98,20.61,government_job,45.244537,MiddleAge,medium,1,Medium


In [34]:
x = dfFeat[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier']]
y = dfFeat['insurance_premium_category']

In [35]:
categoricalFeatures= ['age_group', 'lifestyle_risk', 'occupation', 'city_tier']
numericFeature = ['bmi', 'income_lpa']

In [36]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numericFeature),
        ('cat', OneHotEncoder(), categoricalFeatures)
    ]
)

In [37]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=2))
])

In [38]:
X_train, X_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['bmi', 'income_lpa']),
                                                 ('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation',
                                                   'city_tier'])])),
                ('classifier', RandomForestClassifier(random_state=2))])

In [39]:
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

0.5

In [40]:
import pickle
pickle_model_path = 'model.pkl'
with open(pickle_model_path, 'wb') as  f:
  pickle.dump(pipeline,f)

In [25]:
df['occupation'].unique()

array(['unemployed', 'retired', 'business_owner', 'private_job',
       'student', 'government_job'], dtype=object)